# SmartShop AI — Observability & Performance Analysis

**Red Hat Summit 2026 — Community Proof Bundle**

This notebook pulls live data from:
- Prometheus / Thanos (OCP monitoring) — DCGM GPU metrics, Redis, Spark
- MLflow (RHOAI-hosted) — training runs, cross-run comparisons
- MinIO metrics bundles — post-run JSON artifacts from `collect-run-metrics.sh`

Run cell-by-cell after each Spark/Slurm job to generate publication-quality charts.

**Setup:** all env vars loaded from `../.env`. Run from the repo root or set `REPO_ROOT`.

In [ ]:
import os, json, glob, warnings
from pathlib import Path
from datetime import datetime, timedelta
from dotenv import load_dotenv

warnings.filterwarnings('ignore')

REPO_ROOT = Path(os.environ.get('REPO_ROOT', '..'))
load_dotenv(REPO_ROOT / '.env')

PROM_URL        = os.environ.get('PROMETHEUS_URL', 'https://thanos-querier.openshift-monitoring.svc.cluster.local:9091')
PROM_TOKEN      = os.environ.get('PROMETHEUS_TOKEN', '')  # oc sa get-token grafana-sa -n smartshop
MLFLOW_URI      = os.environ.get('MLFLOW_TRACKING_URI', '')
MINIO_ENDPOINT  = os.environ.get('MINIO_ENDPOINT', '')
AWS_KEY         = os.environ.get('AWS_ACCESS_KEY_ID', '')
AWS_SECRET      = os.environ.get('AWS_SECRET_ACCESS_KEY', '')
S3_BUCKET       = os.environ.get('S3_MODELS_BUCKET', 'smartshop-models')
NAMESPACE       = os.environ.get('NAMESPACE', 'smartshop')

print(f'MLflow: {MLFLOW_URI}')
print(f'Prometheus: {PROM_URL}')
print(f'MinIO: {MINIO_ENDPOINT}')

In [ ]:
# ── Install dependencies if needed ────────────────────────────────────────────
import subprocess, sys
pkgs = ['mlflow', 'prometheus_api_client', 'boto3', 's3fs', 'matplotlib', 'pandas', 'seaborn', 'plotly', 'python-dotenv']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs, check=True)
print('Dependencies ready')

In [ ]:
import ssl, urllib.request, urllib.parse
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import mlflow
import boto3

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 150, 'figure.figsize': (14, 5)})

# Prometheus query helper — works with OCP Thanos (self-signed cert)
def prom_query(promql: str, start=None, end=None, step='15s', instant=False):
    """Query Prometheus/Thanos. Returns a DataFrame with columns: timestamp, value, labels."""
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    headers = {}
    if PROM_TOKEN:
        headers['Authorization'] = f'Bearer {PROM_TOKEN}'

    if instant:
        url = f'{PROM_URL}/api/v1/query?query={urllib.parse.quote(promql)}'
    else:
        end   = end or datetime.utcnow()
        start = start or (end - timedelta(hours=2))
        url = (f'{PROM_URL}/api/v1/query_range'
               f'?query={urllib.parse.quote(promql)}'
               f'&start={start.timestamp():.0f}'
               f'&end={end.timestamp():.0f}'
               f'&step={step}')

    req = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(req, context=ctx, timeout=15) as r:
            data = json.loads(r.read())
    except Exception as e:
        print(f'Prometheus error: {e}')
        return pd.DataFrame()

    results = data.get('data', {}).get('result', [])
    rows = []
    for series in results:
        labels = series['metric']
        values = series.get('values', [series.get('value', [])])
        for ts, val in values:
            rows.append({'timestamp': datetime.fromtimestamp(float(ts)),
                         'value': float(val), **labels})
    return pd.DataFrame(rows)

print('Helpers loaded')

---
## 1. MLflow — GPU vs CPU Speedup Comparison

In [ ]:
if not MLFLOW_URI:
    print('MLFLOW_TRACKING_URI not set — skipping MLflow section')
else:
    mlflow.set_tracking_uri(MLFLOW_URI)
    client = mlflow.tracking.MlflowClient()

    exp = client.get_experiment_by_name('smartshop-feature-engineering')
    if not exp:
        print('Experiment not found — run the Spark jobs first')
    else:
        runs = client.search_runs(
            experiment_ids=[exp.experiment_id],
            order_by=['start_time DESC'],
            max_results=20,
        )
        df_runs = pd.DataFrame([
            {
                'run_name':        r.info.run_name or r.info.run_id[:8],
                'gpu_accelerated': r.data.tags.get('rapids_active', 'False') == 'True',
                'executor_count':  r.data.params.get('executor_instances', '?'),
                'total_elapsed_s': r.data.metrics.get('total_elapsed_s', None),
                'throughput':      r.data.metrics.get('throughput_rows_per_s', None),
                'total_reviews':   r.data.metrics.get('total_reviews', None),
                'status':          r.info.status,
            }
            for r in runs if r.info.status == 'FINISHED'
        ])
        print(df_runs.to_string(index=False))

In [ ]:
# ── GPU vs CPU comparison bar chart ───────────────────────────────────────────
try:
    rapids = df_runs[df_runs.gpu_accelerated == True].iloc[0]
    cpu    = df_runs[df_runs.gpu_accelerated == False].iloc[0]

    speedup = cpu['total_elapsed_s'] / rapids['total_elapsed_s']
    throughput_gain = rapids['throughput'] / cpu['throughput']

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Wall-clock time
    axes[0].bar(['CPU\n(spark-jobs)', 'RAPIDS GPU\n(A100 ×4)'],
                [cpu['total_elapsed_s'], rapids['total_elapsed_s']],
                color=['#6baed6', '#fc8d59'], edgecolor='white', linewidth=1.5)
    axes[0].set_title(f'Feature Engineering Time\n({speedup:.1f}× GPU speedup)', fontsize=13)
    axes[0].set_ylabel('Seconds')
    for i, v in enumerate([cpu['total_elapsed_s'], rapids['total_elapsed_s']]):
        axes[0].text(i, v + 2, f'{v:.0f}s', ha='center', fontweight='bold')

    # Throughput
    axes[1].bar(['CPU', 'RAPIDS GPU'],
                [cpu['throughput'], rapids['throughput']],
                color=['#6baed6', '#fc8d59'], edgecolor='white', linewidth=1.5)
    axes[1].set_title(f'Throughput\n({throughput_gain:.1f}× improvement)', fontsize=13)
    axes[1].set_ylabel('rows/sec')
    for i, v in enumerate([cpu['throughput'], rapids['throughput']]):
        axes[1].text(i, v * 1.02, f'{v:,.0f}', ha='center', fontweight='bold')

    # Speedup gauge
    categories = ['Wall-clock speedup', 'Throughput gain']
    values = [speedup, throughput_gain]
    bars = axes[2].barh(categories, values, color=['#31a354', '#31a354'], height=0.4)
    axes[2].set_title('RAPIDS GPU vs CPU\n(higher = better)', fontsize=13)
    axes[2].set_xlabel('× improvement')
    axes[2].axvline(x=1.0, color='gray', linestyle='--', alpha=0.5, label='CPU baseline')
    for bar, val in zip(bars, values):
        axes[2].text(val + 0.05, bar.get_y() + bar.get_height()/2,
                     f'{val:.1f}×', va='center', fontweight='bold', fontsize=12)

    plt.suptitle(f'SmartShop — Spark Feature Engineering: RAPIDS GPU vs CPU\n'
                 f'Dataset: {cpu["total_reviews"]:,.0f} reviews | Executors: {cpu["executor_count"]}',
                 fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig('/tmp/mlflow_gpu_vs_cpu.png', bbox_inches='tight', dpi=150)
    plt.show()
    print(f'\n  GPU speedup: {speedup:.2f}×  |  Throughput gain: {throughput_gain:.2f}×')
    print(f'  Saved → /tmp/mlflow_gpu_vs_cpu.png')
except Exception as e:
    print(f'Need both CPU and RAPIDS runs in MLflow to draw this chart: {e}')

---
## 2. DCGM GPU Metrics — During RAPIDS Job Window

In [ ]:
# ── Adjust this window to match when your RAPIDS job ran ──────────────────────
# Example: if job ran from 14:00 to 14:22 UTC today
JOB_START = datetime.utcnow() - timedelta(hours=1)   # ← adjust
JOB_END   = datetime.utcnow()                         # ← adjust

print(f'Querying DCGM metrics from {JOB_START} to {JOB_END} UTC')

df_util  = prom_query('DCGM_FI_DEV_GPU_UTIL',      start=JOB_START, end=JOB_END)
df_mem   = prom_query('DCGM_FI_DEV_FB_USED',       start=JOB_START, end=JOB_END)
df_sm    = prom_query('DCGM_FI_PROF_SM_ACTIVE',    start=JOB_START, end=JOB_END)
df_dram  = prom_query('DCGM_FI_PROF_DRAM_ACTIVE',  start=JOB_START, end=JOB_END)
df_power = prom_query('DCGM_FI_DEV_POWER_USAGE',   start=JOB_START, end=JOB_END)
df_nvlink= prom_query('rate(DCGM_FI_DEV_NVLINK_BANDWIDTH_TOTAL[1m])', start=JOB_START, end=JOB_END)

print(f'GPU util samples: {len(df_util)}')

In [ ]:
def plot_dcgm(df, title, ylabel, color='#e6550d', aggregate='mean'):
    if df.empty:
        print(f'No data for {title} — is DCGM exporter running?'); return
    agg = df.groupby('timestamp')['value'].agg(aggregate).reset_index()
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(agg['timestamp'], agg['value'], color=color, linewidth=1.5)
    ax.fill_between(agg['timestamp'], agg['value'], alpha=0.15, color=color)
    ax.set_title(title, fontsize=13)
    ax.set_ylabel(ylabel)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    plt.gcf().autofmt_xdate()
    plt.tight_layout()
    fname = f'/tmp/dcgm_{title.lower().replace(" ","_")[:30]}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  avg={agg["value"].mean():.1f}  max={agg["value"].max():.1f}  → {fname}')

plot_dcgm(df_util,  'GPU Utilization % (RAPIDS job window)', '%',       '#e6550d')
plot_dcgm(df_mem,   'GPU Framebuffer Memory Used (MB)',      'MB',       '#3182bd')
plot_dcgm(df_power, 'GPU Power Usage (W)',                   'Watts',    '#756bb1')
plot_dcgm(df_sm,    'SM Active Ratio',                       'ratio',    '#31a354')
plot_dcgm(df_nvlink,'NVLink Bandwidth (MB/s)',               'MB/s',     '#e7298a')

In [ ]:
# ── Combined 6-panel DCGM figure for blog/slides ──────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
panels = [
    (df_util,   'GPU Utilization (%)',        '%',     '#e6550d'),
    (df_mem,    'GPU Framebuffer Mem (MB)',   'MB',    '#3182bd'),
    (df_sm,     'SM Active Ratio',            '',      '#31a354'),
    (df_dram,   'DRAM Active Ratio',          '',      '#756bb1'),
    (df_power,  'Power (W)',                  'W',     '#e7298a'),
    (df_nvlink, 'NVLink BW (MB/s)',           'MB/s',  '#1c9099'),
]
for ax, (df, title, ylabel, color) in zip(axes.flat, panels):
    if not df.empty:
        agg = df.groupby('timestamp')['value'].mean().reset_index()
        ax.plot(agg['timestamp'], agg['value'], color=color, linewidth=1.2)
        ax.fill_between(agg['timestamp'], agg['value'], alpha=0.15, color=color)
        ax.set_title(title, fontsize=11)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        ax.tick_params(axis='x', labelsize=8)
    else:
        ax.text(0.5, 0.5, 'No data', ha='center', transform=ax.transAxes, color='gray')
        ax.set_title(title, fontsize=11)

plt.suptitle('SmartShop — DCGM GPU Metrics (RAPIDS Feature Engineering job)',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('/tmp/dcgm_combined.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → /tmp/dcgm_combined.png')

---
## 3. Redis Feature Store — Throughput & Hit Ratio

In [ ]:
# Query redis_exporter metrics from user-workload Prometheus
# NOTE: switch PROM_URL to port 9092 for user-workload metrics if not using Thanos
ns_filter = f'{{namespace="{NAMESPACE}"}}'

df_redis_ops  = prom_query(f'rate(redis_commands_processed_total{ns_filter}[1m])', start=JOB_START, end=JOB_END)
df_redis_hits = prom_query(f'rate(redis_keyspace_hits_total{ns_filter}[1m])',     start=JOB_START, end=JOB_END)
df_redis_miss = prom_query(f'rate(redis_keyspace_misses_total{ns_filter}[1m])',   start=JOB_START, end=JOB_END)
df_redis_mem  = prom_query(f'redis_memory_used_bytes{ns_filter}',                 start=JOB_START, end=JOB_END)
df_redis_keys = prom_query(f'redis_db_keys{ns_filter}', instant=True)

print(f'Redis ops samples: {len(df_redis_ops)}')
if not df_redis_keys.empty:
    print(f'Materialized feature keys: {df_redis_keys["value"].sum():,.0f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Commands per second
if not df_redis_ops.empty:
    ops_agg = df_redis_ops.groupby('timestamp')['value'].sum().reset_index()
    axes[0].plot(ops_agg['timestamp'], ops_agg['value'], color='#2ca25f', linewidth=1.5)
    axes[0].fill_between(ops_agg['timestamp'], ops_agg['value'], alpha=0.15, color='#2ca25f')
    axes[0].set_title('Redis Commands/sec\n(Feast online reads during inference)', fontsize=11)
    axes[0].set_ylabel('ops/sec')
else:
    axes[0].text(0.5, 0.5, 'Deploy redis-exporter first', ha='center', transform=axes[0].transAxes, color='gray')

# Cache hit ratio
if not df_redis_hits.empty and not df_redis_miss.empty:
    hits = df_redis_hits.groupby('timestamp')['value'].sum()
    miss = df_redis_miss.groupby('timestamp')['value'].sum()
    ratio = (hits / (hits + miss).replace(0, float('nan'))).dropna().reset_index()
    ratio.columns = ['timestamp', 'hit_ratio']
    axes[1].plot(ratio['timestamp'], ratio['hit_ratio'] * 100, color='#2171b5', linewidth=1.5)
    axes[1].axhline(95, color='green', linestyle='--', alpha=0.6, label='95% target')
    axes[1].set_ylim(0, 105)
    axes[1].set_title('Cache Hit Ratio %\n(target >95% for warm feature store)', fontsize=11)
    axes[1].set_ylabel('%')
    axes[1].legend(fontsize=9)
else:
    axes[1].text(0.5, 0.5, 'No hit/miss data', ha='center', transform=axes[1].transAxes, color='gray')

# Memory used
if not df_redis_mem.empty:
    mem_agg = df_redis_mem.groupby('timestamp')['value'].mean().reset_index()
    mem_agg['value_mb'] = mem_agg['value'] / 1_048_576
    axes[2].plot(mem_agg['timestamp'], mem_agg['value_mb'], color='#756bb1', linewidth=1.5)
    axes[2].fill_between(mem_agg['timestamp'], mem_agg['value_mb'], alpha=0.15, color='#756bb1')
    axes[2].set_title('Redis Memory Used (MB)\n(materialized features footprint)', fontsize=11)
    axes[2].set_ylabel('MB')
else:
    axes[2].text(0.5, 0.5, 'No memory data', ha='center', transform=axes[2].transAxes, color='gray')

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    plt.gcf().autofmt_xdate()

plt.suptitle('SmartShop — Redis Feature Store Metrics', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('/tmp/redis_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → /tmp/redis_metrics.png')

---
## 4. MinIO Metrics Bundle — Spark REST + DCGM Summary

In [ ]:
import s3fs

fs = s3fs.S3FileSystem(
    key=AWS_KEY, secret=AWS_SECRET,
    endpoint_url=MINIO_ENDPOINT, use_ssl=False
)

bundle_files = sorted(fs.glob(f'{S3_BUCKET}/metrics/*_metrics_bundle.json'))
print(f'Found {len(bundle_files)} metric bundles:')
for f in bundle_files:
    print(f'  {f}')

In [ ]:
bundles = []
for f in bundle_files:
    with fs.open(f) as fh:
        bundles.append(json.load(fh))

# Build comparison table from bundles
rows = []
for b in bundles:
    spark = b.get('spark_rest', {})
    dcgm  = b.get('dcgm', {}).get('summary', {})
    ml    = b.get('mlflow', {})
    rows.append({
        'bundle_id':            b.get('bundle_id', '?'),
        'run_type':             b.get('run_type', '?'),
        'collected_at':         b.get('collected_at', '?'),
        'stages':               spark.get('stage_count'),
        'total_task_ms':        spark.get('total_task_time_ms'),
        'shuffle_read_mb':      (spark.get('total_shuffle_read_bytes') or 0) // 1_048_576,
        'shuffle_write_mb':     (spark.get('total_shuffle_write_bytes') or 0) // 1_048_576,
        'gpu_op_count':         spark.get('sql_gpu_operator_count'),
        'cpu_fallback_count':   spark.get('sql_cpu_fallback_count'),
        'gpu_coverage_pct':     spark.get('sql_gpu_coverage_pct'),
        'dcgm_util_avg':        dcgm.get('gpu_utilization_pct_avg'),
        'dcgm_mem_max_mb':      dcgm.get('gpu_memory_used_mb_max'),
        'dcgm_power_avg_w':     dcgm.get('gpu_power_usage_watts_avg'),
        'speedup':              ml.get('gpu_vs_cpu_speedup'),
    })

df_bundles = pd.DataFrame(rows)
df_bundles

In [ ]:
# ── RAPIDS operator coverage pie chart ────────────────────────────────────────
rapids_bundles = df_bundles[df_bundles.run_type == 'rapids'].dropna(subset=['gpu_op_count'])
if not rapids_bundles.empty:
    row = rapids_bundles.iloc[0]
    gpu_ops = int(row['gpu_op_count'] or 0)
    cpu_ops = int(row['cpu_fallback_count'] or 0)

    fig, ax = plt.subplots(1, 2, figsize=(13, 5))

    # Pie chart
    labels = [f'GPU operators ({gpu_ops})', f'CPU fallback ({cpu_ops})']
    colors = ['#fc8d59', '#6baed6']
    wedges, texts, autotexts = ax[0].pie(
        [gpu_ops, cpu_ops], labels=labels, colors=colors,
        autopct='%1.1f%%', startangle=90, pctdistance=0.82,
        wedgeprops=dict(edgecolor='white', linewidth=2)
    )
    [t.set_fontsize(11) for t in autotexts]
    ax[0].set_title('RAPIDS SQL Operator Coverage\n(GPU-accelerated vs CPU fallback)', fontsize=12)

    # GPU vs CPU stage comparison
    if len(df_bundles) >= 2:
        cpu_row    = df_bundles[df_bundles.run_type == 'cpu'].iloc[0]
        rapids_row = df_bundles[df_bundles.run_type == 'rapids'].iloc[0]
        categories = ['Total task time (ms)', 'Shuffle read (MB)', 'Shuffle write (MB)']
        cpu_vals    = [cpu_row['total_task_ms'] or 0, cpu_row['shuffle_read_mb'] or 0, cpu_row['shuffle_write_mb'] or 0]
        rapids_vals = [rapids_row['total_task_ms'] or 0, rapids_row['shuffle_read_mb'] or 0, rapids_row['shuffle_write_mb'] or 0]
        x = range(len(categories))
        width = 0.35
        ax[1].bar([i - width/2 for i in x], cpu_vals,    width, label='CPU',         color='#6baed6')
        ax[1].bar([i + width/2 for i in x], rapids_vals, width, label='RAPIDS GPU',  color='#fc8d59')
        ax[1].set_xticks(list(x)); ax[1].set_xticklabels(categories, fontsize=10)
        ax[1].set_title('Spark Stage Metrics: CPU vs RAPIDS', fontsize=12)
        ax[1].legend()

    plt.tight_layout()
    plt.savefig('/tmp/rapids_coverage.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'GPU coverage: {row["gpu_coverage_pct"]}%  |  Saved → /tmp/rapids_coverage.png')
else:
    print('Run RAPIDS job and collect-run-metrics.sh to generate this chart')

---
## 5. MLflow Stage Breakdown — Per-Operation Timing

In [ ]:
# Compare per-stage timings: user features / item features / interactions
try:
    rapids_run = next(r for r in runs if r.data.tags.get('rapids_active') == 'True')
    cpu_run    = next(r for r in runs if r.data.tags.get('rapids_active') == 'False')

    stages = ['user_features', 'item_features', 'interactions']
    cpu_times    = [cpu_run.data.metrics.get(f'{s}_elapsed_s', 0) for s in stages]
    rapids_times = [rapids_run.data.metrics.get(f'{s}_elapsed_s', 0) for s in stages]

    x = range(len(stages))
    width = 0.35
    fig, ax = plt.subplots(figsize=(11, 5))
    b1 = ax.bar([i - width/2 for i in x], cpu_times,    width, label='CPU', color='#6baed6')
    b2 = ax.bar([i + width/2 for i in x], rapids_times, width, label='RAPIDS GPU', color='#fc8d59')
    ax.set_xticks(list(x))
    ax.set_xticklabels([s.replace('_', ' ').title() for s in stages])
    ax.set_ylabel('Seconds')
    ax.set_title('Per-Stage Elapsed Time: CPU vs RAPIDS GPU\n'
                 '(groupBy+agg, join, interactions — all GPU-accelerated by RAPIDS)', fontsize=12)
    ax.legend()

    for bar in [*b1, *b2]:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{bar.get_height():.0f}s', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.savefig('/tmp/mlflow_stage_breakdown.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved → /tmp/mlflow_stage_breakdown.png')
except StopIteration:
    print('Need both CPU and RAPIDS MLflow runs')

---
## 6. Export — Bundle all charts for blog / community post

In [ ]:
import shutil, zipfile

chart_files = [
    '/tmp/mlflow_gpu_vs_cpu.png',
    '/tmp/dcgm_combined.png',
    '/tmp/redis_metrics.png',
    '/tmp/rapids_coverage.png',
    '/tmp/mlflow_stage_breakdown.png',
]

zip_path = '/tmp/smartshop_summit_charts.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for f in chart_files:
        if Path(f).exists():
            zf.write(f, Path(f).name)
            print(f'  Added: {Path(f).name}')

print(f'\nBundle: {zip_path}')

# Optionally upload to MinIO for sharing
try:
    with fs.open(f'{S3_BUCKET}/metrics/summit_charts.zip', 'wb') as f_out:
        with open(zip_path, 'rb') as f_in:
            f_out.write(f_in.read())
    print(f'Uploaded → s3://{S3_BUCKET}/metrics/summit_charts.zip')
except Exception as e:
    print(f'MinIO upload skipped: {e}')

---
## Appendix: Key PromQL Queries for OCP Observe Console

Paste these directly into OCP → Observe → Metrics:

```promql
# GPU utilization heatmap (all GPUs in cluster)
DCGM_FI_DEV_GPU_UTIL

# GPU memory pressure
DCGM_FI_DEV_FB_USED / (DCGM_FI_DEV_FB_USED + DCGM_FI_DEV_FB_FREE)

# Redis ops/sec (Feast online store traffic)
rate(redis_commands_processed_total{namespace="smartshop"}[1m])

# Redis hit ratio (feature freshness)
rate(redis_keyspace_hits_total{namespace="smartshop"}[5m]) /
  (rate(redis_keyspace_hits_total{namespace="smartshop"}[5m]) +
   rate(redis_keyspace_misses_total{namespace="smartshop"}[5m]))

# Spark executor GC pressure
rate(metrics_jvm_total_gc_time_ms{namespace="smartshop"}[1m])

# GPU power consumption (W) — useful for cost per inference estimate
DCGM_FI_DEV_POWER_USAGE

# NVLink bandwidth (proves GPU-to-GPU shuffle is via NVLink not PCIe)
rate(DCGM_FI_DEV_NVLINK_BANDWIDTH_TOTAL[1m])
```

## OpenTelemetry — When to Add It

| Scenario | Recommendation |
|---|---|
| Redis access latency p99 | `redis_exporter` is sufficient — it uses `LATENCY HISTORY` internally |
| Feast → Redis → KServe distributed traces | OTEL Collector + Tempo/Jaeger needed; too heavy for Summit scope |
| MinIO S3 data access timing | Boto3 event hooks in `download.py` are simpler than OTEL SDK |
| Training metrics (loss, throughput) | MLflow is already OTEL-compatible; no additional SDK needed |
| Future: full request trace from Gradio → KServe → Feast → Redis | Add `opentelemetry-instrumentation-requests` + OTEL Collector + Tempo |

The `redis_exporter` covers **all Redis data access proof** you need for this demo.
OTEL full tracing is a natural next step once the pipeline is stable post-Summit.